. Trình bày trong bài thuyết trình
Slide 1: Lý thuyết Gradient Boosting

Gradient Boosting là một thuật toán học máy dựa trên cây quyết định, xây dựng mô hình bằng cách kết hợp nhiều cây yếu (weak learners).
Mỗi cây mới được thêm vào để giảm thiểu sai số của mô hình hiện tại bằng cách tối ưu hóa hàm mất mát (loss function).
Ưu điểm:
Hiệu suất cao với dữ liệu phi tuyến tính.
Khả năng tự động chọn lọc đặc trưng.
Nhược điểm:
Cần tinh chỉnh siêu tham số.
Tốn thời gian huấn luyện hơn so với các mô hình đơn giản.
Slide 2: Quy trình thực nghiệm

Bước 1: Chuẩn bị dữ liệu (xử lý dữ liệu, mã hóa, chia tập).
Bước 2: Huấn luyện mô hình với XGBoost.
Bước 3: Đánh giá mô hình bằng các metrics (MAE, RMSE, R²).
Bước 4: So sánh kết quả với các mô hình khác.
Slide 3: Kết quả và nhận xét

Trình bày bảng hoặc biểu đồ so sánh các metrics giữa Gradient Boosting và các mô hình khác.
Giải thích tại sao Gradient Boosting hoạt động tốt (hoặc không tốt) với dữ liệu của bạn.


In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import KFold, train_test_split, RandomizedSearchCV
import json
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import joblib
import matplotlib.pyplot as plt

# Hàm Target Encoding có KFold
def target_encode_kfold(df, column, target, n_fold=5, alpha=5):
    """
    Performs target encoding with k-fold cross-validation to prevent data leakage.
    
    Parameters:
    -----------
    df : pandas DataFrame
        The dataframe containing the data
    column : str
        The name of the categorical column to encode
    target : str
        The name of the target column
    n_fold : int, default=5
        Number of folds for cross-validation
    alpha : float, default=5
        Regularization parameter
        
    Returns:
    --------
    pandas Series
        The encoded values for the column
    """
    # Create a copy of the dataframe
    df_copy = df.copy()
    
    # Calculate the global mean
    global_mean = df_copy[target].mean()
    
    # Create the output Series
    encoded = pd.Series(index=df_copy.index)
    
    # Split the data into n_fold parts
    kf = KFold(n_splits=n_fold, shuffle=True, random_state=42)
    
    for train_idx, test_idx in kf.split(df_copy):
        # Calculate the mean target for each category in the training fold
        means = df_copy.iloc[train_idx].groupby(column)[target].agg(['mean', 'count'])
        
        # Apply regularization
        smoothed_means = ((means['mean'] * means['count']) + (global_mean * alpha)) / (means['count'] + alpha)
        
        # Map the means to the test fold
        encoded.iloc[test_idx] = df_copy.iloc[test_idx][column].map(smoothed_means)
    
    # Handle any categories that weren't seen in the training data (use global mean)
    encoded.fillna(global_mean, inplace=True)
    
    return encoded

In [ ]:
# Bước 1: Đọc dữ liệu
train_file = '../../data/datasets/train_stratified_by_chip_model.csv'
test_file = '../../data/datasets/test_stratified_by_chip_model.csv'

train_df = pd.read_csv(train_file)
test_df = pd.read_csv(test_file)

# Các cột cần encoding và cột mục tiêu
categorical_cols = ['chip_model', 'brand', 'screen_tech', 'screen_resolution_k']
target_col = 'price'

# Lưu lại mean của các categorical features
encoding_means = {}
encoding_means_path = './datasets/encoding_means.json'


In [ ]:
# Bước 2: Target Encoding
for col in categorical_cols:
    print(f"Encoding {col}...")
    train_df[f'{col}_encoded'] = target_encode_kfold(
        train_df, column=col, target=target_col, n_fold=5, alpha=10
    )
    # Lưu mean của từng category
    encoding_means[col] = train_df.groupby(col)[f'{col}_encoded'].mean().to_dict()

# Lưu encoding_means vào file JSON
with open(encoding_means_path, 'w') as f:
    json.dump(encoding_means, f)
print(f"Saved encoding means to ${encoding_means_path}")

# Áp dụng encoding cho tập test
for col in categorical_cols:
    test_df[f'{col}_encoded'] = test_df[col].map(encoding_means[col]).fillna(train_df[target_col].mean())



In [ ]:
# Bước 3: Chuẩn bị dữ liệu
features = ['ram', 'internal_memory', 'screen_size', 'refresh_rate', 
            'camera_main_resolution', 'camera_count', 'chip_model_encoded', 
            'brand_encoded', 'screen_tech_encoded', 'screen_resolution_k_encoded']
target = 'price'

X_train = train_df[features]
y_train = train_df[target_col]
X_test = test_df[features]
y_test = test_df[target_col]

In [ ]:
# Bước 4: Tối ưu hóa siêu tham số
param_dist = {
    'n_estimators': [100, 300, 500, 600, 800],
    'learning_rate': [0.01, 0.005, 0.001],
    'max_depth': [3, 5, 6, 8],
    'subsample': [0.6, 0.8, 1.0],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 3, 5]
}

gb_base = GradientBoostingRegressor(random_state=42)

random_search = RandomizedSearchCV(
    gb_base,
    param_distributions=param_dist,
    n_iter=30,
    cv=3,
    verbose=2,
    n_jobs=-1,
    scoring='neg_mean_absolute_error',
    random_state=42
)

print("Tuning hyperparameters...")
random_search.fit(X_train, y_train)
best_model = random_search.best_estimator_

print("✅ Best Hyperparameters:")
print(random_search.best_params_)

In [ ]:
# Bước 6: Lưu model
print("Evaluating model...")
y_pred = best_model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print(f"MAE: {mae}")
print(f"RMSE: {rmse}")
print(f"R²: {r2}")

# Lưu model tốt nhất
print("Saving best model...")
joblib.dump(best_model, 'best_gb_model.pkl')
print("✅ Saved best model to 'best_gb_model.pkl'")

# Bước 7: Trực quan hóa loss trên tập train
plt.figure(figsize=(10, 6))
plt.plot(np.arange(1, len(best_model.train_score_) + 1), best_model.train_score_, label='Train Loss', color='blue')
plt.xlabel('Number of Estimators')
plt.ylabel('MSE')
plt.title('Train Loss Curve')
plt.legend()
plt.grid()
plt.show()

Test

In [ ]:
import pandas as pd
import joblib
import json

# Đường dẫn đến file trọng số và encoding means
model_file = 'best_gb_model.pkl'
encoding_means_file = './datasets/encoding_means.json'

# Tải mô hình đã lưu
model = joblib.load(model_file)
print("Model loaded successfully!")

# Tải encoding means
with open(encoding_means_file, 'r') as f:
    encoding_means = json.load(f)
print("Encoding means loaded successfully!")

# Đọc tập test
test_file = '../../data/datasets/test_stratified_by_chip_model.csv'
test_df = pd.read_csv(test_file)

# Các cột cần mã hóa
categorical_cols = ['chip_model', 'brand', 'screen_tech', 'screen_resolution_k']

# Áp dụng encoding cho tập test
for col in categorical_cols:
    test_df[f'{col}_encoded'] = test_df[col].map(encoding_means[col]).fillna(0)

# Các cột đặc trưng
features = ['ram', 'internal_memory', 'screen_size', 'refresh_rate', 
            'camera_main_resolution', 'camera_count', 'chip_model_encoded', 
            'brand_encoded', 'screen_tech_encoded', 'screen_resolution_k_encoded']

# Chuẩn bị dữ liệu
X_test = test_df[features]

# Dự đoán
y_pred = model.predict(X_test)

# Lưu kết quả dự đoán
test_df['predicted_price'] = y_pred
output_file = 'test_predictions.csv'
test_df.to_csv(output_file, index=False)
print(f"Predictions saved to {output_file}")

### 1. Trực quan hóa hiệu suất mô hình
- a. Biểu đồ so sánh giá trị thực tế và giá trị dự đoán
    - Mục đích: Để kiểm tra mức độ khớp giữa giá trị thực tế (y_test) và giá trị dự đoán (y_pred).
    - Cách thực hiện:

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Biểu đồ scatter giữa giá trị thực tế và giá trị dự đoán
plt.figure(figsize=(8, 8))
plt.scatter(y_test, y_pred, alpha=0.5)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
plt.xlabel('Giá trị thực tế')
plt.ylabel('Giá trị dự đoán')
plt.title('So sánh giá trị thực tế và giá trị dự đoán')
plt.grid()
plt.show()

b. Biểu đồ phân phối sai số (Residual Plot)
- Mục đích: Để kiểm tra xem sai số có phân phối ngẫu nhiên hay không.
- Cách thực hiện:

In [ ]:
# Tính residuals
residuals = y_test - y_pred

# Biểu đồ phân phối residuals
plt.figure(figsize=(8, 6))
plt.hist(residuals, bins=30, alpha=0.7, color='blue', edgecolor='black')
plt.axvline(0, color='red', linestyle='--', linewidth=2)
plt.xlabel('Residuals')
plt.ylabel('Tần suất')
plt.title('Phân phối sai số (Residuals)')
plt.grid()
plt.show()

### 2. Trực quan hóa tầm quan trọng của đặc trưng (Feature Importance)
- Mục đích: Để hiểu đặc trưng nào đóng vai trò quan trọng nhất trong việc dự đoán.
- Cách thực hiện:

In [ ]:
# Lấy tầm quan trọng của đặc trưng từ mô hình
feature_importances = model.feature_importances_
feature_names = features

# Biểu đồ cột cho tầm quan trọng của đặc trưng
plt.figure(figsize=(10, 6))
plt.barh(feature_names, feature_importances, color='skyblue')
plt.xlabel('Tầm quan trọng')
plt.ylabel('Đặc trưng')
plt.title('Tầm quan trọng của các đặc trưng')
plt.grid(axis='x')
plt.show()

### 3. Trực quan hóa mối quan hệ giữa đặc trưng và giá trị mục tiêu
- a. Biểu đồ scatter giữa đặc trưng quan trọng nhất và giá trị mục tiêu
    - Mục đích: Để kiểm tra mối quan hệ giữa đặc trưng quan trọng nhất và giá trị mục tiêu (price).
    - Cách thực hiện:

In [ ]:
# Chọn đặc trưng quan trọng nhất
most_important_feature = feature_names[np.argmax(feature_importances)]

# Biểu đồ scatter
plt.figure(figsize=(8, 6))
plt.scatter(train_df[most_important_feature], train_df[target], alpha=0.5, color='green')
plt.xlabel(most_important_feature)
plt.ylabel('Price')
plt.title(f'Mối quan hệ giữa {most_important_feature} và Price')
plt.grid()
plt.show()

### 4. Trực quan hóa phân phối giá trị thực tế và giá trị dự đoán
- Mục đích: Để kiểm tra xem phân phối của giá trị dự đoán có khớp với giá trị thực tế hay không.
- Cách thực hiện:

In [ ]:
# Biểu đồ phân phối giá trị thực tế và giá trị dự đoán
plt.figure(figsize=(8, 6))
plt.hist(y_test, bins=30, alpha=0.5, label='Giá trị thực tế', color='blue', edgecolor='black')
plt.hist(y_pred, bins=30, alpha=0.5, label='Giá trị dự đoán', color='orange', edgecolor='black')
plt.xlabel('Price')
plt.ylabel('Tần suất')
plt.title('Phân phối giá trị thực tế và giá trị dự đoán')
plt.legend()
plt.grid()
plt.show()

# Lasso Regression Implementation

Lasso (Least Absolute Shrinkage and Selection Operator) là một phương pháp hồi quy tuyến tính sử dụng kỹ thuật điều chuẩn L1.
Ưu điểm của Lasso:

-   Thực hiện lựa chọn biến tự động bằng cách đưa một số hệ số về 0
-   Giảm overfitting
-   Xử lý tốt hiện tượng đa cộng tuyến (multicollinearity)

Trong phần này, chúng ta sẽ huấn luyện mô hình Lasso và so sánh với mô hình Gradient Boosting.


In [ ]:
![image.png](attachment:image.png)

**Tên mô hình: Gradient Boosting (GBM)**

- Là mô hình ensemble kết hợp nhiều cây quyết định để cải thiện dự đoán qua từng bước.
- Tại mỗi bước boosting, mô hình mới học theo **gradient âm của hàm mất mát** để sửa lỗi từ mô hình trước.

Phân tích các hệ số (coefficients) của mô hình Lasso để xem các đặc trưng nào quan trọng và đặc trưng nào bị loại bỏ (hệ số = 0).


In [ ]:
# Hiển thị các hệ số của mô hình Lasso
coefs = pd.DataFrame({
    'Feature': features,
    'Coefficient': lasso_model.coef_
})

# Sắp xếp theo giá trị tuyệt đối của hệ số để thấy đặc trưng quan trọng nhất
coefs['Absolute_Coefficient'] = np.abs(coefs['Coefficient'])
coefs = coefs.sort_values(by='Absolute_Coefficient', ascending=False)

# Số đặc trưng bị loại bỏ (hệ số = 0)
zero_coefs = sum(coefs['Coefficient'] == 0)
print(f"Số đặc trưng bị loại bỏ (hệ số = 0): {zero_coefs} / {len(features)}")

print("\nTop các đặc trưng quan trọng:")
display(coefs.head())

# Trực quan hóa hệ số
plt.figure(figsize=(12, 8))
coefs = coefs.sort_values(by='Coefficient')
colors = ['red' if c < 0 else 'blue' for c in coefs['Coefficient']]
plt.barh(coefs['Feature'], coefs['Coefficient'], color=colors)
plt.title('Hệ số của các đặc trưng trong mô hình Lasso')
plt.xlabel('Coefficient Value')
plt.grid(axis='x')
plt.show()

### 2. So sánh mô hình Lasso với Gradient Boosting

Biểu đồ so sánh giá trị thực tế và dự đoán giữa hai mô hình.


In [ ]:
# Lấy dự đoán từ mô hình Gradient Boosting nếu có
try:
    gb_model = joblib.load('best_gb_model.pkl')
    y_pred_gb = gb_model.predict(X_test)
    
    # Tính metrics cho Gradient Boosting
    mae_gb = mean_absolute_error(y_test, y_pred_gb)
    rmse_gb = np.sqrt(mean_squared_error(y_test, y_pred_gb))
    r2_gb = r2_score(y_test, y_pred_gb)
    
    # So sánh metrics
    metrics_comparison = pd.DataFrame({
        'Model': ['Lasso Regression', 'Gradient Boosting'],
        'MAE': [mae_lasso, mae_gb],
        'RMSE': [rmse_lasso, rmse_gb],
        'R²': [r2_lasso, r2_gb]
    })
    
    display(metrics_comparison)
    
    # Trực quan hóa so sánh dự đoán
    plt.figure(figsize=(12, 8))
    
    plt.subplot(1, 2, 1)
    plt.scatter(y_test, y_pred_lasso, alpha=0.5)
    plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
    plt.xlabel('Giá trị thực tế')
    plt.ylabel('Giá trị dự đoán')
    plt.title('Lasso Regression')
    plt.grid()
    
    plt.subplot(1, 2, 2)
    plt.scatter(y_test, y_pred_gb, alpha=0.5)
    plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
    plt.xlabel('Giá trị thực tế')
    plt.ylabel('Giá trị dự đoán')
    plt.title('Gradient Boosting')
    plt.grid()
    
    plt.tight_layout()
    plt.show()
    
except Exception as e:
    print(f"Không thể tải mô hình Gradient Boosting: {e}")
    
    # Chỉ hiển thị kết quả của Lasso
    metrics_lasso = pd.DataFrame({
        'Model': ['Lasso Regression'],
        'MAE': [mae_lasso],
        'RMSE': [rmse_lasso],
        'R²': [r2_lasso]
    })
    
    display(metrics_lasso)
    
    # Chỉ vẽ biểu đồ cho Lasso
    plt.figure(figsize=(8, 8))
    plt.scatter(y_test, y_pred_lasso, alpha=0.5)
    plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
    plt.xlabel('Giá trị thực tế')
    plt.ylabel('Giá trị dự đoán')
    plt.title('Lasso Regression: Actual vs Predicted')
    plt.grid()
    plt.show()

# Random Forest Regression Implementation

Random Forest là một thuật toán học máy dựa trên kỹ thuật ensemble learning, kết hợp nhiều cây quyết định để tạo ra một mô hình mạnh mẽ hơn.

Ưu điểm của Random Forest:

- Khả năng xử lý dữ liệu có chiều cao, không cần feature scaling
- Không dễ bị overfitting nhờ tính ngẫu nhiên trong quá trình xây dựng mô hình
- Cung cấp thông tin về tầm quan trọng của các đặc trưng
- Hoạt động tốt với cả dữ liệu phân loại và hồi quy

Trong phần này, chúng ta sẽ huấn luyện mô hình Random Forest và so sánh với các mô hình đã triển khai trước đó.

In [ ]:
# Import thư viện
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json  # Thêm dòng này để đảm bảo json được import
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import joblib
import seaborn as sns

# Đọc dữ liệu đã mã hóa (sử dụng lại dữ liệu đã chuẩn bị từ trước)
try:
    train_file = '../../data/datasets/train_stratified_by_chip_model.csv'
    test_file = '../../data/datasets/test_stratified_by_chip_model.csv'

    print(f"Đang đọc dữ liệu từ: {train_file} và {test_file}")
    train_df = pd.read_csv(train_file)
    test_df = pd.read_csv(test_file)
    print(f"Đã đọc dữ liệu thành công, train shape: {train_df.shape}, test shape: {test_df.shape}")
except Exception as e:
    print(f"Lỗi khi đọc dữ liệu: {e}")

# Các cột cần encoding và cột mục tiêu
categorical_cols = ['chip_model', 'brand', 'screen_tech', 'screen_resolution_k']
target_col = 'price'

# Lấy encoding_means từ file nếu có hoặc tạo mới
try:
    encoding_means_path = './datasets/encoding_means.json'
    with open(encoding_means_path, 'r') as f:
        encoding_means = json.load(f)
    print("Loaded existing encoding means")
    
    # Áp dụng encoding từ file đã có
    for col in categorical_cols:
        if f'{col}_encoded' not in train_df.columns:
            train_df[f'{col}_encoded'] = train_df[col].map(encoding_means[col]).fillna(train_df[target_col].mean())
        if f'{col}_encoded' not in test_df.columns:
            test_df[f'{col}_encoded'] = test_df[col].map(encoding_means[col]).fillna(train_df[target_col].mean())
except Exception as e:
    print(f"Không thể tải encoding means: {e}")
    # Nếu chưa có encoding, thực hiện lại quá trình encoding
    print("Tạo encoding mới từ hàm target_encode_kfold")
    encoding_means = {}
    for col in categorical_cols:
        print(f"Encoding {col}...")
        # Sử dụng hàm đã được định nghĩa trước đó trong notebook
        train_df[f'{col}_encoded'] = target_encode_kfold(
            train_df, column=col, target=target_col, n_fold=5, alpha=10
        )
        # Lưu mean của từng category
        encoding_means[col] = train_df.groupby(col)[f'{col}_encoded'].mean().to_dict()
        # Áp dụng encoding cho tập test
        test_df[f'{col}_encoded'] = test_df[col].map(encoding_means[col]).fillna(train_df[target_col].mean())
    
    # Lưu encoding_means vào file JSON
    try:
        # Tạo thư mục nếu chưa tồn tại
        import os
        os.makedirs('./datasets', exist_ok=True)
        
        with open('./datasets/encoding_means.json', 'w') as f:
            json.dump(encoding_means, f)
        print("Đã lưu encoding means vào file JSON")
    except Exception as e:
        print(f"Không thể lưu encoding means: {e}")

# Các cột đặc trưng và biến mục tiêu
features = ['ram', 'internal_memory', 'screen_size', 'refresh_rate', 
            'camera_main_resolution', 'camera_count', 'chip_model_encoded', 
            'brand_encoded', 'screen_tech_encoded', 'screen_resolution_k_encoded']
target = 'price'

print(f"Features: {features}")

# Chuẩn bị dữ liệu
X_train = train_df[features]
y_train = train_df[target]
X_test = test_df[features]
y_test = test_df[target]

# Huấn luyện mô hình Random Forest
try:
    print("Bắt đầu huấn luyện mô hình Random Forest...")
    rf_model = RandomForestRegressor(
        n_estimators=100,  # Số lượng cây
        max_depth=10,      # Độ sâu tối đa của mỗi cây
        min_samples_split=5,  # Số lượng mẫu tối thiểu để phân tách một nút
        min_samples_leaf=2,   # Số lượng mẫu tối thiểu ở mỗi lá
        random_state=42,
        n_jobs=-1,  # Sử dụng tất cả các cpu cores
        bootstrap=True,  # Sử dụng bootstrap samples
        oob_score=True,  # Tính điểm Out-of-Bag để đánh giá mô hình
    )

    # Huấn luyện mô hình
    rf_model.fit(X_train, y_train)
    print("Đã huấn luyện mô hình Random Forest thành công")
    
    # In điểm Out-of-Bag (OOB) - một cách kiểm tra hiệu suất mà không cần tập validation
    print(f"OOB Score: {rf_model.oob_score_:.4f}")

    # Lưu mô hình vào file
    joblib.dump(rf_model, 'random_forest_model.pkl')
    print("Saved model to 'random_forest_model.pkl'")

    # Dự đoán trên tập test
    y_pred_rf = rf_model.predict(X_test)

    # Đánh giá mô hình
    mae_rf = mean_absolute_error(y_test, y_pred_rf)
    rmse_rf = np.sqrt(mean_squared_error(y_test, y_pred_rf))
    r2_rf = r2_score(y_test, y_pred_rf)
    mape_rf = np.mean(np.abs((y_test - y_pred_rf) / y_test)) * 100  # MAPE (Mean Absolute Percentage Error)

    print(f"Mean Absolute Error (MAE): {mae_rf:.2f}")
    print(f"Root Mean Squared Error (RMSE): {rmse_rf:.2f}")
    print(f"R² Score: {r2_rf:.4f}")
    print(f"Mean Absolute Percentage Error (MAPE): {mape_rf:.2f}%")
except Exception as e:
    print(f"Lỗi khi huấn luyện mô hình Random Forest: {e}")

In [ ]:
from sklearn.model_selection import RandomizedSearchCV, GridSearchCV
from sklearn.ensemble import RandomForestRegressor
import numpy as np
import time

# Định nghĩa lưới siêu tham số cho RandomizedSearchCV
param_dist = {
    'n_estimators': [100, 200, 300, 400, 500],
    'max_depth': [10, 20, 30, 40, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': ['auto', 'sqrt', 'log2'],
    'bootstrap': [True, False]
}

# Tạo mô hình cơ bản
rf = RandomForestRegressor(random_state=42, n_jobs=-1, oob_score=True)

# Bắt đầu đo thời gian
start_time = time.time()

# Tìm kiếm siêu tham số tốt nhất
random_search = RandomizedSearchCV(
    estimator=rf,
    param_distributions=param_dist,
    n_iter=30,
    cv=3,
    verbose=2,
    random_state=42,
    n_jobs=-1,
    scoring='r2'  # Sử dụng R² làm thước đo để tối ưu
)

# Huấn luyện mô hình với RandomizedSearchCV
print("🔍 Bắt đầu tìm kiếm siêu tham số tốt nhất với RandomizedSearchCV...")
random_search.fit(X_train, y_train)
print("✅ Tìm kiếm hoàn tất.")

# Tính thời gian đã trôi qua
elapsed_time = time.time() - start_time
print(f"⏱️ Thời gian tìm kiếm: {elapsed_time:.2f} giây ({elapsed_time/60:.2f} phút)")

# Hiển thị tham số tốt nhất
print("\n🎯 Siêu tham số tốt nhất:")
for param, value in random_search.best_params_.items():
    print(f"{param}: {value}")

# Hiển thị điểm tốt nhất
print(f"\nĐiểm R² tốt nhất trên tập validation (cross-validation): {random_search.best_score_:.4f}")

# Lưu mô hình tốt nhất
best_model = random_search.best_estimator_
joblib.dump(best_model, 'random_forest_best.pkl')
print("💾 Đã lưu mô hình tốt nhất vào 'random_forest_best.pkl'")

# Dự đoán và đánh giá mô hình tốt nhất
y_pred_best = best_model.predict(X_test)

# Đánh giá toàn diện
mae_best = mean_absolute_error(y_test, y_pred_best)
rmse_best = np.sqrt(mean_squared_error(y_test, y_pred_best))
r2_best = r2_score(y_test, y_pred_best)
mape_best = np.mean(np.abs((y_test - y_pred_best) / y_test)) * 100

print(f"\n📈 Đánh giá mô hình tốt nhất:")
print(f"MAE: {mae_best:.2f}")
print(f"RMSE: {rmse_best:.2f}")
print(f"R² Score: {r2_best:.4f}")
print(f"MAPE: {mape_best:.2f}%")

# So sánh với mô hình cơ bản
improvement_mae = ((mae_rf - mae_best) / mae_rf) * 100
improvement_rmse = ((rmse_rf - rmse_best) / rmse_rf) * 100
improvement_r2 = ((r2_best - r2_rf) / abs(r2_rf)) * 100

print("\n📊 Cải thiện so với mô hình cơ bản:")
print(f"MAE: Giảm {improvement_mae:.2f}%")
print(f"RMSE: Giảm {improvement_rmse:.2f}%")
print(f"R²: Tăng {improvement_r2:.2f}%")

# Mức độ quan trọng của siêu tham số
if hasattr(random_search, 'cv_results_'):
    # Tạo DataFrame từ kết quả tìm kiếm
    results = pd.DataFrame(random_search.cv_results_)
    
    # Lọc các cột param và mean_test_score
    param_cols = [col for col in results.columns if col.startswith('param_')]
    score_col = 'mean_test_score'
    
    # Tạo danh sách correlation giữa param và score
    correlations = []
    
    for param in param_cols:
        # Bỏ qua các tham số không phải số
        param_values = results[param].values
        if not all(isinstance(v, (int, float)) or v is None for v in param_values):
            continue
            
        # Thay thế None bằng -1 để tính toán correlation
        param_values = [-1 if v is None else v for v in param_values]
        
        # Tính correlation với score
        correlation = np.corrcoef(param_values, results[score_col])[0, 1]
        correlations.append((param.replace('param_', ''), abs(correlation)))
    
    # Sắp xếp theo độ quan trọng
    correlations.sort(key=lambda x: x[1], reverse=True)
    
    print("\n🔑 Mức độ ảnh hưởng của siêu tham số (dựa trên tương quan):")
    for param, corr in correlations:
        print(f"{param}: {corr:.4f}")
    
    # Vẽ biểu đồ so sánh các lần thử
    plt.figure(figsize=(10, 6))
    plt.scatter(range(len(results)), results['mean_test_score'], alpha=0.7)
    plt.axhline(y=random_search.best_score_, color='r', linestyle='--', label=f'Best score: {random_search.best_score_:.4f}')
    plt.xlabel('Lần thử')
    plt.ylabel('Điểm R²')
    plt.title('Kết quả các lần thử trong RandomizedSearchCV')
    plt.legend()
    plt.grid(True)
    plt.savefig('rf_randomized_search_results.png', dpi=300)
    plt.show()


## Trực quan hóa và phân tích kết quả của Random Forest

### 1. Phân tích tầm quan trọng của đặc trưng và giải thích mô hình

In [ ]:
# Hiển thị tầm quan trọng của đặc trưng từ cả hai mô hình Random Forest
# (mô hình cơ bản và mô hình tốt nhất sau khi tinh chỉnh siêu tham số)

# Tạo DataFrame để hiển thị
feature_importance_comparison = pd.DataFrame({
    'Feature': features,
    'Basic RF Importance': rf_model.feature_importances_,
    'Best RF Importance': best_model.feature_importances_
})

# Sắp xếp theo tầm quan trọng của mô hình tốt nhất
feature_importance_comparison = feature_importance_comparison.sort_values('Best RF Importance', ascending=False)

# Hiển thị kết quả
print("So sánh tầm quan trọng của đặc trưng giữa hai mô hình:")
display(feature_importance_comparison)

# Tính % thay đổi trong tầm quan trọng
feature_importance_comparison['Change (%)'] = ((feature_importance_comparison['Best RF Importance'] - 
                                            feature_importance_comparison['Basic RF Importance']) / 
                                           feature_importance_comparison['Basic RF Importance']) * 100

# Hiển thị kết quả với % thay đổi
print("\nPhần trăm thay đổi trong tầm quan trọng của đặc trưng sau khi tinh chỉnh:")
display(feature_importance_comparison[['Feature', 'Basic RF Importance', 'Best RF Importance', 'Change (%)']])

# Trực quan hóa tầm quan trọng của đặc trưng - so sánh hai mô hình
plt.figure(figsize=(12, 8))

# Vẽ biểu đồ cột nhóm để so sánh
x = np.arange(len(features))
width = 0.35

fig, ax = plt.subplots(figsize=(14, 8))
bars1 = ax.barh(x - width/2, feature_importance_comparison['Basic RF Importance'], 
               width, label='Basic RF', color='skyblue', edgecolor='black')
bars2 = ax.barh(x + width/2, feature_importance_comparison['Best RF Importance'], 
               width, label='Best RF', color='purple', edgecolor='black')

# Thêm chi tiết vào biểu đồ
ax.set_xlabel('Tầm quan trọng', fontsize=12)
ax.set_title('So sánh tầm quan trọng của đặc trưng giữa hai mô hình Random Forest', fontsize=14)
ax.set_yticks(x)
ax.set_yticklabels(feature_importance_comparison['Feature'])
ax.legend()
ax.grid(axis='x')

# Thêm nhãn giá trị cho từng cột
for i, bar in enumerate(bars1):
    width = bar.get_width()
    ax.text(width + 0.005, bar.get_y() + bar.get_height()/2, f"{width:.4f}", 
            ha='left', va='center', fontsize=9)

for i, bar in enumerate(bars2):
    width = bar.get_width()
    ax.text(width + 0.005, bar.get_y() + bar.get_height()/2, f"{width:.4f}", 
            ha='left', va='center', fontsize=9)

plt.tight_layout()
plt.savefig('rf_feature_importance_comparison.png', dpi=300)
plt.show()

# Trực quan hóa phân phối cây quyết định (Tree Decision) cho mô hình tốt nhất
# Chỉ hiển thị một số cây đầu tiên để minh họa
try:
    from sklearn.tree import plot_tree
    
    # Lấy 2 cây đầu tiên từ Random Forest tốt nhất
    n_trees_to_show = 2
    figsize = (18, 10)
    
    for i in range(min(n_trees_to_show, len(best_model.estimators_))):
        plt.figure(figsize=figsize)
        tree = best_model.estimators_[i]
        plot_tree(tree, 
                 feature_names=features, 
                 filled=True, 
                 max_depth=3,  # Giới hạn độ sâu để dễ quan sát
                 proportion=True)
        plt.title(f'Cây quyết định #{i+1} trong Random Forest', fontsize=14)
        plt.tight_layout()
        plt.savefig(f'rf_decision_tree_{i+1}.png', dpi=300)
        plt.show()
except ImportError:
    print("Không thể import plot_tree từ sklearn.tree, bỏ qua bước trực quan hóa cây quyết định.")
except Exception as e:
    print(f"Lỗi khi trực quan hóa cây quyết định: {e}")

### 2. Trực quan hóa giá trị thực tế và giá trị dự đoán với phân tích chi tiết sai số

In [ ]:
# Trực quan hóa kết quả dự đoán cả hai mô hình Random Forest
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Vẽ cho mô hình cơ bản
axes[0].scatter(y_test, y_pred_rf, alpha=0.5, color='skyblue')
axes[0].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
axes[0].set_xlabel('Giá trị thực tế')
axes[0].set_ylabel('Giá trị dự đoán')
axes[0].set_title('Random Forest cơ bản: Thực tế vs Dự đoán')
axes[0].grid(True)

# Vẽ cho mô hình tốt nhất
axes[1].scatter(y_test, y_pred_best, alpha=0.5, color='purple')
axes[1].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
axes[1].set_xlabel('Giá trị thực tế')
axes[1].set_ylabel('Giá trị dự đoán')
axes[1].set_title('Best Random Forest: Thực tế vs Dự đoán')
axes[1].grid(True)

plt.tight_layout()
plt.savefig('rf_models_actual_vs_pred.png', dpi=300)
plt.show()

# Biểu đồ phân phối sai số (Residuals) của cả hai mô hình
residuals_rf = y_test - y_pred_rf
residuals_best = y_test - y_pred_best

plt.figure(figsize=(12, 6))
plt.hist(residuals_rf, bins=30, alpha=0.5, label='Random Forest cơ bản', color='skyblue', edgecolor='black')
plt.hist(residuals_best, bins=30, alpha=0.5, label='Best Random Forest', color='purple', edgecolor='black')
plt.axvline(0, color='red', linestyle='--', linewidth=2)
plt.xlabel('Sai số (Residuals)')
plt.ylabel('Tần suất')
plt.title('So sánh phân phối sai số giữa hai mô hình Random Forest')
plt.legend()
plt.grid(True)
plt.savefig('rf_models_residuals.png', dpi=300)
plt.show()

# Phân tích sai số chi tiết theo giá trị dự đoán
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Vẽ residual plot cho mô hình cơ bản
axes[0].scatter(y_pred_rf, residuals_rf, alpha=0.5, color='skyblue')
axes[0].axhline(y=0, color='red', linestyle='--', linewidth=2)
axes[0].set_xlabel('Giá trị dự đoán')
axes[0].set_ylabel('Sai số (Residuals)')
axes[0].set_title('Random Forest cơ bản: Residual Plot')
axes[0].grid(True)

# Vẽ residual plot cho mô hình tốt nhất
axes[1].scatter(y_pred_best, residuals_best, alpha=0.5, color='purple')
axes[1].axhline(y=0, color='red', linestyle='--', linewidth=2)
axes[1].set_xlabel('Giá trị dự đoán')
axes[1].set_ylabel('Sai số (Residuals)')
axes[1].set_title('Best Random Forest: Residual Plot')
axes[1].grid(True)

plt.tight_layout()
plt.savefig('rf_models_residual_plots.png', dpi=300)
plt.show()

# Phân tích sai số theo khoảng giá
# Chia giá trị thành các nhóm (phân vị)
price_bins = pd.qcut(y_test, 4, labels=['Thấp', 'Trung bình thấp', 'Trung bình cao', 'Cao'])

# Tạo DataFrame sai số theo nhóm giá
error_by_price = pd.DataFrame({
    'Actual': y_test,
    'Predicted_Basic': y_pred_rf,
    'Predicted_Best': y_pred_best,
    'Error_Basic': np.abs(y_test - y_pred_rf),
    'Error_Best': np.abs(y_test - y_pred_best),
    'Error_Pct_Basic': np.abs((y_test - y_pred_rf) / y_test) * 100,
    'Error_Pct_Best': np.abs((y_test - y_pred_best) / y_test) * 100,
    'Price_Range': price_bins
})

# Tính sai số trung bình theo nhóm giá
error_summary = error_by_price.groupby('Price_Range').agg({
    'Error_Basic': 'mean',
    'Error_Best': 'mean',
    'Error_Pct_Basic': 'mean',
    'Error_Pct_Best': 'mean'
})

print("Sai số trung bình theo khoảng giá:")
display(error_summary)

# Trực quan hóa sai số theo khoảng giá
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Vẽ sai số tuyệt đối
error_summary[['Error_Basic', 'Error_Best']].plot(kind='bar', ax=axes[0])
axes[0].set_title('Sai số tuyệt đối theo khoảng giá')
axes[0].set_ylabel('Sai số trung bình')
axes[0].grid(axis='y')

# Vẽ sai số phần trăm
error_summary[['Error_Pct_Basic', 'Error_Pct_Best']].plot(kind='bar', ax=axes[1])
axes[1].set_title('Sai số phần trăm theo khoảng giá')
axes[1].set_ylabel('Sai số phần trăm (%)')
axes[1].grid(axis='y')

plt.tight_layout()
plt.savefig('rf_error_by_price_range.png', dpi=300)
plt.show()

### 3. So sánh chi tiết các mô hình (Random Forest, Gradient Boosting, và Lasso)

In [ ]:
# So sánh các mô hình
try:
    # Tải các mô hình đã huấn luyện nếu có
    models_metrics = {
        'Random Forest (Cơ bản)': {'MAE': mae_rf, 'RMSE': rmse_rf, 'R²': r2_rf, 'MAPE (%)': mape_rf},
        'Random Forest (Tốt nhất)': {'MAE': mae_best, 'RMSE': rmse_best, 'R²': r2_best, 'MAPE (%)': mape_best}
    }
    
    # Thêm metrics của Gradient Boosting nếu có
    try:
        gb_model = joblib.load('best_gb_model.pkl')
        y_pred_gb = gb_model.predict(X_test)
        mae_gb = mean_absolute_error(y_test, y_pred_gb)
        rmse_gb = np.sqrt(mean_squared_error(y_test, y_pred_gb))
        r2_gb = r2_score(y_test, y_pred_gb)
        mape_gb = np.mean(np.abs((y_test - y_pred_gb) / y_test)) * 100
        models_metrics['Gradient Boosting'] = {'MAE': mae_gb, 'RMSE': rmse_gb, 'R²': r2_gb, 'MAPE (%)': mape_gb}
    except Exception as e:
        print(f"Không thể tải mô hình Gradient Boosting: {e}")
    
    # Thêm metrics của Lasso nếu có
    try:
        lasso_model = joblib.load('lasso_model.pkl')
        scaler = joblib.load('scaler_for_lasso.pkl')
        X_test_scaled = scaler.transform(X_test)
        y_pred_lasso = lasso_model.predict(X_test_scaled)
        mae_lasso = mean_absolute_error(y_test, y_pred_lasso)
        rmse_lasso = np.sqrt(mean_squared_error(y_test, y_pred_lasso))
        r2_lasso = r2_score(y_test, y_pred_lasso)
        mape_lasso = np.mean(np.abs((y_test - y_pred_lasso) / y_test)) * 100
        models_metrics['Lasso Regression'] = {'MAE': mae_lasso, 'RMSE': rmse_lasso, 'R²': r2_lasso, 'MAPE (%)': mape_lasso}
    except Exception as e:
        print(f"Không thể tải mô hình Lasso: {e}")
    
    # Tạo DataFrame so sánh
    comparison_df = pd.DataFrame.from_dict(models_metrics, orient='index')
    display(comparison_df)
    
    # Định dạng DataFrame với các metrics được làm tròn
    formatted_df = comparison_df.copy()
    formatted_df['MAE'] = formatted_df['MAE'].round(2)
    formatted_df['RMSE'] = formatted_df['RMSE'].round(2)
    formatted_df['R²'] = formatted_df['R²'].round(4)
    formatted_df['MAPE (%)'] = formatted_df['MAPE (%)'].round(2)
    
    # Sắp xếp theo R² giảm dần
    formatted_df = formatted_df.sort_values('R²', ascending=False)
    
    # Hiển thị bảng đã định dạng
    print("\nXếp hạng các mô hình theo R² Score (cao đến thấp):")
    display(formatted_df)
    
    # Tạo bảng xếp hạng các mô hình cho từng metric (1 = tốt nhất)
    rank_df = comparison_df.copy()
    rank_df['MAE_rank'] = rank_df['MAE'].rank()
    rank_df['RMSE_rank'] = rank_df['RMSE'].rank()
    rank_df['R²_rank'] = rank_df['R²'].rank(ascending=False)
    rank_df['MAPE_rank'] = rank_df['MAPE (%)'].rank()
    
    # Tính điểm trung bình
    rank_df['Avg_Rank'] = rank_df[['MAE_rank', 'RMSE_rank', 'R²_rank', 'MAPE_rank']].mean(axis=1)
    
    # Sắp xếp theo điểm trung bình
    rank_df = rank_df.sort_values('Avg_Rank')
    
    # Hiển thị bảng xếp hạng
    print("\nXếp hạng các mô hình theo từng metric (1 = tốt nhất):")
    display(rank_df[['MAE_rank', 'RMSE_rank', 'R²_rank', 'MAPE_rank', 'Avg_Rank']])
    
    # Lấy mô hình tốt nhất dựa trên điểm trung bình
    best_model_name = rank_df.index[0]
    print(f"\n🏆 Mô hình tốt nhất dựa trên điểm trung bình: {best_model_name}")
    
    # Trực quan hóa so sánh các metrics
    metrics = ['MAE', 'RMSE', 'R²', 'MAPE (%)']
    models = list(models_metrics.keys())
    
    plt.figure(figsize=(15, 10))
    
    for i, metric in enumerate(metrics):
        plt.subplot(2, 2, i+1)
        values = [models_metrics[model][metric] for model in models]
        
        # Màu sắc cho từng mô hình
        colors = ['skyblue', 'purple', 'lightgreen', 'coral'][:len(models)]
        if 'Random Forest (Tốt nhất)' in models:
            idx = models.index('Random Forest (Tốt nhất)')
            colors[idx] = 'purple'  # Đảm bảo Best RF luôn màu tím
            
        plt.bar(models, values, color=colors)
        plt.title(f'So sánh {metric}', fontsize=12)
        plt.xticks(rotation=45, ha='right')
        plt.grid(axis='y')
        
        # Đánh dấu giá trị trên mỗi cột
        for j, v in enumerate(values):
            if metric == 'R²':
                plt.text(j, v, f"{v:.4f}", ha='center', va='bottom', fontweight='bold')
            else:
                plt.text(j, v, f"{v:.2f}", ha='center', va='bottom', fontweight='bold')
    
    plt.tight_layout()
    plt.savefig('detailed_model_comparison.png', dpi=300)
    plt.show()
    
except Exception as e:
    print(f"Lỗi khi so sánh các mô hình: {e}")

### 4. Phân tích thời gian đáp ứng và độ phức tạp của mô hình (tradeoff)

In [ ]:
# Phân tích hiệu suất và thời gian đáp ứng của các mô hình
import time

# Danh sách các mô hình để so sánh thời gian dự đoán
models_to_compare = {}

try:
    # Thêm mô hình Random Forest (cơ bản và tốt nhất)
    models_to_compare['Random Forest (Cơ bản)'] = rf_model
    models_to_compare['Random Forest (Tốt nhất)'] = best_model
    
    # Thêm mô hình Gradient Boosting nếu có
    try:
        gb_model = joblib.load('best_gb_model.pkl')
        models_to_compare['Gradient Boosting'] = gb_model
    except Exception as e:
        print(f"Không thể tải mô hình Gradient Boosting: {e}")
    
    # Thêm mô hình Lasso nếu có
    try:
        lasso_model = joblib.load('lasso_model.pkl')
        scaler = joblib.load('scaler_for_lasso.pkl')
        # Không thêm trực tiếp vì cần transform dữ liệu trước
    except Exception as e:
        print(f"Không thể tải mô hình Lasso: {e}")
    
    # So sánh thời gian đáp ứng
    timing_results = []
    n_runs = 10  # Số lần chạy để lấy trung bình
    
    for model_name, model in models_to_compare.items():
        # Đo thời gian dự đoán
        start_time = time.time()
        
        for _ in range(n_runs):
            _ = model.predict(X_test)
            
        end_time = time.time()
        avg_time = (end_time - start_time) / n_runs
        
        # Lấy thông tin về độ phức tạp của mô hình
        if hasattr(model, 'n_estimators'):
            complexity = model.n_estimators
            complexity_type = 'số cây'
        else:
            complexity = None
            complexity_type = 'không xác định'
        
        # Thêm vào kết quả
        timing_results.append({
            'Model': model_name,
            'Avg Prediction Time (s)': avg_time,
            'Complexity': complexity,
            'Complexity Type': complexity_type
        })
    
    # Thêm mô hình Lasso nếu có
    try:
        # Đo thời gian dự đoán cho Lasso
        start_time = time.time()
        
        for _ in range(n_runs):
            X_test_scaled = scaler.transform(X_test)
            _ = lasso_model.predict(X_test_scaled)
            
        end_time = time.time()
        avg_time = (end_time - start_time) / n_runs
        
        # Lấy thông tin về độ phức tạp của mô hình (số hệ số khác 0)
        non_zero_coefs = np.sum(lasso_model.coef_ != 0)
        
        # Thêm vào kết quả
        timing_results.append({
            'Model': 'Lasso Regression',
            'Avg Prediction Time (s)': avg_time,
            'Complexity': non_zero_coefs,
            'Complexity Type': 'số hệ số khác 0'
        })
    except Exception as e:
        print(f"Không thể đo thời gian cho mô hình Lasso: {e}")
    
    # Tạo DataFrame từ kết quả
    timing_df = pd.DataFrame(timing_results)
    
    # Hiển thị kết quả
    print("\nSo sánh thời gian dự đoán và độ phức tạp của các mô hình:")
    display(timing_df)
    
    # Tạo biểu đồ so sánh thời gian dự đoán
    plt.figure(figsize=(10, 6))
    
    # Sắp xếp theo thời gian tăng dần
    timing_df_sorted = timing_df.sort_values('Avg Prediction Time (s)')
    
    # Vẽ biểu đồ cột
    colors = ['skyblue', 'purple', 'lightgreen', 'coral'][:len(timing_df_sorted)]
    bars = plt.barh(timing_df_sorted['Model'], timing_df_sorted['Avg Prediction Time (s)'], color=colors)
    
    # Thêm nhãn
    plt.xlabel('Thời gian dự đoán trung bình (giây)', fontsize=12)
    plt.title('So sánh thời gian dự đoán của các mô hình', fontsize=14)
    plt.grid(axis='x')
    
    # Thêm giá trị lên mỗi cột
    for bar in bars:
        width = bar.get_width()
        plt.text(width + 0.0005, bar.get_y() + bar.get_height()/2, f"{width:.5f}s", 
                ha='left', va='center', fontweight='bold')
    
    plt.tight_layout()
    plt.savefig('model_prediction_time.png', dpi=300)
    plt.show()
    
    # Trực quan hóa tradeoff giữa hiệu suất (R²) và thời gian dự đoán
    plt.figure(figsize=(10, 6))
    
    # Lấy R² score từ các mô hình
    r2_values = []
    for model_name in timing_df['Model']:
        if model_name in models_metrics:
            r2_values.append(models_metrics[model_name]['R²'])
        else:
            r2_values.append(None)
    
    # Thêm R² vào DataFrame
    timing_df['R² Score'] = r2_values
    
    # Lọc các hàng có R² score
    timing_df_filtered = timing_df.dropna(subset=['R² Score'])
    
    # Vẽ biểu đồ scatter
    for i, row in timing_df_filtered.iterrows():
        plt.scatter(row['Avg Prediction Time (s)'], row['R² Score'], 
                   s=100, label=row['Model'])
        plt.annotate(row['Model'], 
                    (row['Avg Prediction Time (s)'], row['R² Score']),
                    xytext=(5, 5), textcoords='offset points')
    
    plt.xlabel('Thời gian dự đoán (giây)', fontsize=12)
    plt.ylabel('R² Score', fontsize=12)
    plt.title('Tradeoff giữa hiệu suất và thời gian dự đoán', fontsize=14)
    plt.grid(True)
    plt.tight_layout()
    plt.savefig('model_performance_time_tradeoff.png', dpi=300)
    plt.show()
    
except Exception as e:
    print(f"Lỗi khi phân tích thời gian đáp ứng: {e}")

## Nhận xét và kết luận về Random Forest

Qua việc triển khai và phân tích toàn diện mô hình Random Forest, chúng ta có thể rút ra một số nhận xét quan trọng:

1. **Về hiệu suất mô hình:**
   - Mô hình Random Forest tốt nhất (đã tinh chỉnh siêu tham số) mang lại hiệu suất đáng kể so với mô hình cơ bản
   - Random Forest thường cung cấp sự cân bằng tốt giữa độ chính xác và thời gian huấn luyện
   - Điểm R² cao cho thấy mô hình có khả năng giải thích tốt sự biến động trong giá điện thoại

2. **Về đặc trưng quan trọng:**
   - Các đặc trưng quan trọng nhất ảnh hưởng đến giá điện thoại thường bao gồm RAM, bộ nhớ trong và chip_model
   - Sau khi tinh chỉnh, có sự thay đổi trong thứ tự quan trọng của các đặc trưng, cho thấy vai trò của siêu tham số

3. **Về sai số dự đoán:**
   - Mô hình Random Forest có khuynh hướng dự đoán tốt hơn trong khoảng giá trung bình
   - Điện thoại ở phân khúc giá cao thường có sai số phần trăm thấp hơn, chứng tỏ mô hình hoạt động tốt với dữ liệu phức tạp

4. **Về thời gian đáp ứng:**
   - Tuy Random Forest có thời gian dự đoán dài hơn so với các mô hình tuyến tính như Lasso, nhưng sự tradeoff giữa hiệu suất và thời gian là hợp lý
   - Mô hình tối ưu có thể đạt được bằng cách cân nhắc giữa số lượng cây và độ phức tạp

5. **So sánh với các mô hình khác:**
   - Random Forest cạnh tranh tốt với Gradient Boosting về mặt hiệu suất
   - So với Lasso, Random Forest có khả năng bắt các mối quan hệ phi tuyến tính giữa các đặc trưng và giá trị mục tiêu

6. **Kết luận:**
   - Random Forest là một lựa chọn tốt cho bài toán dự đoán giá điện thoại do khả năng xử lý dữ liệu phức tạp, đặc trưng phi tuyến và robustness trước outliers
   - Việc tinh chỉnh siêu tham số là bước quan trọng để cải thiện hiệu suất của mô hình Random Forest
   - Đối với ứng dụng thực tế, nên cân nhắc sử dụng Random Forest đã tinh chỉnh nếu độ chính xác là ưu tiên, hoặc mô hình đơn giản hơn nếu tốc độ dự đoán là quan trọng

## Trực quan hóa Best Model Random Forest

Trong phần này, chúng ta sẽ nạp và phân tích mô hình Random Forest tốt nhất đã được lưu ở file `random_forest_best.pkl`. Mô hình này đã được tinh chỉnh siêu tham số để có hiệu suất tối ưu.

In [ ]:
# Nạp Best Model Random Forest
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Đường dẫn đến best model
best_rf_model_path = 'random_forest_best.pkl'

try:
    # Nạp best model
    best_rf_model = joblib.load(best_rf_model_path)
    print(f"Đã nạp best Random Forest model thành công từ: {best_rf_model_path}")
    
    # Kiểm tra các tham số của model
    print("\nCác siêu tham số của Best Random Forest Model:")
    for param, value in best_rf_model.get_params().items():
        if param in ['n_estimators', 'max_depth', 'min_samples_split', 'min_samples_leaf', 'max_features', 'bootstrap']:
            print(f"{param}: {value}")
    
    # Sử dụng dữ liệu test để đánh giá model
    y_pred_best_rf = best_rf_model.predict(X_test)
    
    # Tính toán các metrics đánh giá
    mae_best_rf = mean_absolute_error(y_test, y_pred_best_rf)
    rmse_best_rf = np.sqrt(mean_squared_error(y_test, y_pred_best_rf))
    r2_best_rf = r2_score(y_test, y_pred_best_rf)
    
    print("\nMetrics của Best Random Forest Model:")
    print(f"Mean Absolute Error (MAE): {mae_best_rf:.2f}")
    print(f"Root Mean Squared Error (RMSE): {rmse_best_rf:.2f}")
    print(f"R² Score: {r2_best_rf:.4f}")
    
except Exception as e:
    print(f"Lỗi khi nạp best Random Forest model: {e}")

### 1. So sánh dự đoán của Best Random Forest với giá trị thực tế

In [ ]:
try:
    # Trực quan hóa giá trị thực tế và giá trị dự đoán
    plt.figure(figsize=(10, 8))
    plt.scatter(y_test, y_pred_best_rf, alpha=0.5, color='purple')
    plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
    plt.xlabel('Giá trị thực tế', fontsize=12)
    plt.ylabel('Giá trị dự đoán', fontsize=12)
    plt.title('Best Random Forest: Giá trị thực tế vs Giá trị dự đoán', fontsize=14)
    plt.grid(True)
    plt.tight_layout()
    plt.savefig('best_rf_actual_vs_pred.png', dpi=300)
    plt.show()
    
    # Phân phối sai số (Residuals)
    residuals_best_rf = y_test - y_pred_best_rf
    
    plt.figure(figsize=(10, 6))
    sns.histplot(residuals_best_rf, kde=True, color='purple')
    plt.axvline(0, color='red', linestyle='--', linewidth=2)
    plt.xlabel('Sai số (Residuals)', fontsize=12)
    plt.ylabel('Tần suất', fontsize=12)
    plt.title('Phân phối sai số của Best Random Forest', fontsize=14)
    plt.grid(True)
    plt.tight_layout()
    plt.savefig('best_rf_residuals.png', dpi=300)
    plt.show()
    
    # Residual Plot (Sai số so với giá trị dự đoán)
    plt.figure(figsize=(10, 6))
    plt.scatter(y_pred_best_rf, residuals_best_rf, alpha=0.5, color='purple')
    plt.axhline(y=0, color='red', linestyle='--', linewidth=2)
    plt.xlabel('Giá trị dự đoán', fontsize=12)
    plt.ylabel('Sai số (Residuals)', fontsize=12)
    plt.title('Residual Plot của Best Random Forest', fontsize=14)
    plt.grid(True)
    plt.tight_layout()
    plt.savefig('best_rf_residual_plot.png', dpi=300)
    plt.show()
except Exception as e:
    print(f"Lỗi khi trực quan hóa kết quả Best Random Forest: {e}")

### 2. Tầm quan trọng của các đặc trưng trong Best Random Forest Model

In [ ]:
try:
    # Tầm quan trọng của các đặc trưng
    best_feature_importances = best_rf_model.feature_importances_
    
    # Tạo DataFrame để hiển thị
    best_feature_importance_df = pd.DataFrame({
        'Feature': features,
        'Importance': best_feature_importances
    })
    
    # Sắp xếp theo tầm quan trọng giảm dần
    best_feature_importance_df = best_feature_importance_df.sort_values('Importance', ascending=False)
    
    # Hiển thị kết quả
    print("Tầm quan trọng của đặc trưng trong Best Random Forest:")
    display(best_feature_importance_df)
    
    # Trực quan hóa tầm quan trọng của đặc trưng
    plt.figure(figsize=(12, 8))
    ax = sns.barplot(x='Importance', y='Feature', data=best_feature_importance_df, palette='viridis')
    
    # Thêm giá trị lên mỗi cột
    for i, v in enumerate(best_feature_importance_df['Importance']):
        ax.text(v + 0.01, i, f"{v:.4f}", va='center', fontweight='bold')
        
    plt.xlabel('Tầm quan trọng', fontsize=12)
    plt.ylabel('Đặc trưng', fontsize=12)
    plt.title('Tầm quan trọng của các đặc trưng trong Best Random Forest', fontsize=14)
    plt.tight_layout()
    plt.savefig('best_rf_feature_importance.png', dpi=300)
    plt.show()
except Exception as e:
    print(f"Lỗi khi trực quan hóa tầm quan trọng của đặc trưng: {e}")

### 3. So sánh toàn diện tất cả các mô hình đã huấn luyện

In [ ]:
try:
    # Tạo dictionary chứa metrics của tất cả các mô hình
    all_models_metrics = {
        'Best Random Forest': {'MAE': mae_best_rf, 'RMSE': rmse_best_rf, 'R²': r2_best_rf}
    }
    
    # Thêm Random Forest model ban đầu nếu có
    try:
        all_models_metrics['Random Forest'] = {'MAE': mae_rf, 'RMSE': rmse_rf, 'R²': r2_rf}
    except NameError:
        # Nếu các biến không tồn tại, bỏ qua
        print("Random Forest metrics không có sẵn")
    
    # Thêm metrics của Gradient Boosting nếu có
    try:
        all_models_metrics['Gradient Boosting'] = {'MAE': mae_gb, 'RMSE': rmse_gb, 'R²': r2_gb}
    except NameError:
        try:
            # Nếu chưa có sẵn, thử tải mô hình và tính toán
            gb_model = joblib.load('gradient_boosting_model.pkl')
            y_pred_gb = gb_model.predict(X_test)
            mae_gb = mean_absolute_error(y_test, y_pred_gb)
            rmse_gb = np.sqrt(mean_squared_error(y_test, y_pred_gb))
            r2_gb = r2_score(y_test, y_pred_gb)
            all_models_metrics['Gradient Boosting'] = {'MAE': mae_gb, 'RMSE': rmse_gb, 'R²': r2_gb}
        except Exception as e:
            print(f"Không thể tải mô hình Gradient Boosting: {e}")
    
    # Thêm metrics của Lasso nếu có
    try:
        all_models_metrics['Lasso Regression'] = {'MAE': mae_lasso, 'RMSE': rmse_lasso, 'R²': r2_lasso}
    except NameError:
        try:
            # Nếu chưa có sẵn, thử tải mô hình và tính toán
            lasso_model = joblib.load('lasso_model.pkl')
            scaler = joblib.load('scaler_for_lasso.pkl')
            X_test_scaled = scaler.transform(X_test)
            y_pred_lasso = lasso_model.predict(X_test_scaled)
            mae_lasso = mean_absolute_error(y_test, y_pred_lasso)
            rmse_lasso = np.sqrt(mean_squared_error(y_test, y_pred_lasso))
            r2_lasso = r2_score(y_test, y_pred_lasso)
            all_models_metrics['Lasso Regression'] = {'MAE': mae_lasso, 'RMSE': rmse_lasso, 'R²': r2_lasso}
        except Exception as e:
            print(f"Không thể tải mô hình Lasso: {e}")
    
    # Thử tải mô hình RandomForest đã lưu
    try:
        if 'Random Forest' not in all_models_metrics:
            rf_standard_model = joblib.load('random_forest_model.pkl')
            y_pred_rf_std = rf_standard_model.predict(X_test)
            mae_rf_std = mean_absolute_error(y_test, y_pred_rf_std)
            rmse_rf_std = np.sqrt(mean_squared_error(y_test, y_pred_rf_std))
            r2_rf_std = r2_score(y_test, y_pred_rf_std)
            all_models_metrics['Random Forest'] = {'MAE': mae_rf_std, 'RMSE': rmse_rf_std, 'R²': r2_rf_std}
    except Exception as e:
        print(f"Không thể tải mô hình Random Forest: {e}")
            
    # Tạo DataFrame so sánh
    all_comparison_df = pd.DataFrame.from_dict(all_models_metrics, orient='index')
    print("\nSo sánh toàn diện các mô hình:")
    display(all_comparison_df)
    
    # Sắp xếp các mô hình theo R²
    sorted_df = all_comparison_df.sort_values(by='R²', ascending=False)
    print("\nXếp hạng mô hình theo R² (cao đến thấp):")
    display(sorted_df)
    
    # Trực quan hóa so sánh các metrics
    metrics = ['MAE', 'RMSE', 'R²']
    models = list(all_models_metrics.keys())
    
    # Bảng màu cho các mô hình
    colors = {
        'Best Random Forest': 'purple', 
        'Random Forest': 'skyblue', 
        'Gradient Boosting': 'lightgreen', 
        'Lasso Regression': 'coral'
    }
    
    # Biểu đồ cột so sánh tất cả các metrics
    plt.figure(figsize=(15, 6))
    
    for i, metric in enumerate(metrics):
        plt.subplot(1, 3, i+1)
        values = [all_models_metrics[model][metric] for model in models if model in all_models_metrics]
        model_colors = [colors.get(model, 'gray') for model in models if model in all_models_metrics]
        plt.bar(models, values, color=model_colors)
        plt.title(f'So sánh {metric}', fontsize=12)
        plt.xticks(rotation=45, ha='right')
        plt.grid(axis='y')
        
        # Đánh dấu giá trị trên mỗi cột
        for j, v in enumerate(values):
            plt.text(j, v, f"{v:.4f}", ha='center', va='bottom', fontweight='bold')
    
    plt.tight_layout()
    plt.savefig('all_models_comparison.png', dpi=300)
    plt.show()
    
    # Radar chart để so sánh các mô hình
    from math import pi
    
    # Số lượng metrics trong radar chart (angles là các góc)
    labels = list(metrics)
    num_vars = len(labels)
    angles = np.linspace(0, 2*pi, num_vars, endpoint=False).tolist()
    angles += angles[:1]  # Đóng vòng tròn
    
    # Chuẩn hóa các giá trị metrics để có thể so sánh
    # R² giữ nguyên (càng cao càng tốt)
    # MAE, RMSE: đảo ngược (càng thấp càng tốt -> 1 - normalized(value))
    
    # Lấy giá trị lớn nhất và nhỏ nhất của mỗi metric
    max_values = all_comparison_df.max()
    min_values = all_comparison_df.min()
    
    # Tạo bản sao của DataFrame để chuẩn hóa
    normalized_df = all_comparison_df.copy()
    
    # Chuẩn hóa từng metric
    for metric in ['MAE', 'RMSE']:
        if max_values[metric] != min_values[metric]:  # Tránh chia cho 0
            # Chuẩn hóa và đảo ngược giá trị (1 - normalized) vì MAE và RMSE càng thấp càng tốt
            normalized_df[metric] = 1 - ((all_comparison_df[metric] - min_values[metric]) / 
                                       (max_values[metric] - min_values[metric]))
        else:
            normalized_df[metric] = 0.5  # Nếu tất cả các giá trị bằng nhau
    
    # Chuẩn hóa R² (R² càng cao càng tốt)
    if max_values['R²'] != min_values['R²']:
        normalized_df['R²'] = (all_comparison_df['R²'] - min_values['R²']) / (max_values['R²'] - min_values['R²'])
    else:
        normalized_df['R²'] = 0.5
    
    # Thiết lập biểu đồ radar
    plt.figure(figsize=(10, 10))
    ax = plt.subplot(111, polar=True)
    
    # Thêm các trục metrics
    plt.xticks(angles[:-1], labels, fontsize=12)
    
    # Vẽ cho từng mô hình
    for i, model in enumerate(models):
        if model in normalized_df.index:
            values = normalized_df.loc[model].values.tolist()
            values += values[:1]  # Đóng vòng tròn
            
            ax.plot(angles, values, linewidth=2, linestyle='solid', label=model, color=colors.get(model, 'gray'))
            ax.fill(angles, values, alpha=0.1, color=colors.get(model, 'gray'))
    
    # Thêm chi tiết cho biểu đồ
    plt.title('So sánh Radar Chart các mô hình', size=15)
    plt.legend(loc='upper right', bbox_to_anchor=(0.1, 0.1))
    plt.tight_layout()
    plt.savefig('radar_chart_comparison.png', dpi=300)
    plt.show()
    
except Exception as e:
    print(f"Lỗi khi so sánh các mô hình: {e}")

## Kết luận về Random Forest Best Model

Qua phân tích trực quan và so sánh mô hình Random Forest tốt nhất với các mô hình khác, chúng ta có thể rút ra các kết luận sau:

1. **Hiệu suất mô hình**:
   - Random Forest tốt nhất có thể đạt được R² Score cao hơn so với mô hình Random Forest thông thường
   - Việc tinh chỉnh siêu tham số giúp cải thiện đáng kể khả năng dự đoán

2. **Đặc trưng quan trọng**:
   - Mô hình đã xác định được các đặc trưng quan trọng nhất ảnh hưởng đến giá điện thoại
   - Thông số kỹ thuật như RAM, bộ nhớ trong và chip_model có tác động lớn nhất đến giá

3. **So sánh với các mô hình khác**:
   - Random Forest tốt nhất có thể cạnh tranh hoặc vượt trội hơn Gradient Boosting trong một số trường hợp
   - Lasso Regression, mặc dù đơn giản hơn, thường có hiệu suất thấp hơn với dữ liệu phi tuyến tính

4. **Ứng dụng thực tế**:
   - Mô hình có thể được triển khai để dự đoán giá điện thoại mới dựa trên thông số
   - Có thể kết hợp các mô hình để tạo ra một mô hình tổng hợp (ensemble) mạnh hơn

Random Forest tốt nhất cung cấp sự cân bằng tốt giữa độ chính xác và khả năng giải thích, làm cho nó trở thành một lựa chọn hấp dẫn cho bài toán dự đoán giá điện thoại.